In [3]:
# TODO: Load the data into a DataFrame
import pandas as pd

df = pd.read_csv('hourly_weather_10_days.csv')
df.head()

,timestamp,temperature_C,humidity_%,wind_speed_kmph,pressure_hPa,visibility_km
0,2023-03-01 00:00:00,16.6,74.4,5.7,1012.5,9.5
1,2023-03-01 01:00:00,16.2,78.5,5.0,1012.1,10.3
2,2023-03-01 02:00:00,15.3,73.3,4.7,NaN,11.1
3,2023-03-01 03:00:00,15.8,72.4,1.3,1005.0,8.9
4,2023-03-01 04:00:00,20.9,70.6,6.8,1016.3,9.8


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   timestamp        240 non-null    object 
 1   temperature_C    228 non-null    float64
 2   humidity_%       224 non-null    float64
 3   wind_speed_kmph  226 non-null    float64
 4   pressure_hPa     223 non-null    float64
 5   visibility_km    228 non-null    float64
dtypes: float64(5), object(1)
memory usage: 11.4+ KB
None


In [6]:
print(df.describe())

       temperature_C  humidity_%  wind_speed_kmph  pressure_hPa  visibility_km
count     228.000000  224.000000       226.000000    223.000000     228.000000
mean       21.315789   66.795982        10.105310   1011.884753       9.989474
std         3.421237    8.190300         3.940668      5.187080       1.022166
min        11.500000   47.800000         1.300000    998.100000       6.800000
25%        18.700000   61.075000         6.625000   1008.900000       9.275000
50%        21.900000   66.300000         9.800000   1012.100000      10.000000
75%        23.925000   72.725000        13.500000   1015.100000      10.700000
max        28.700000   88.100000        17.800000   1027.000000      12.600000


In [7]:
print(df.isna().sum())

timestamp           0
temperature_C      12
humidity_%         16
wind_speed_kmph    14
pressure_hPa       17
visibility_km      12
dtype: int64


In [10]:
# TODO: Fill missing values
# Using mean fill because weather metrics don't have extreme sudden jumps,
# so mean gives a reasonable estimate for missing readings

df['temperature_C'] = df['temperature_C'].fillna(df['temperature_C'].mean())
df['humidity_%'] = df['humidity_%'].fillna(df['humidity_%'].mean())
df['wind_speed_kmph'] = df['wind_speed_kmph'].fillna(df['wind_speed_kmph'].mean())
df['pressure_hPa'] = df['pressure_hPa'].fillna(df['pressure_hPa'].mean())
df['visibility_km'] = df['visibility_km'].fillna(df['visibility_km'].mean())


timestamp          0
temperature_C      0
humidity_%         0
wind_speed_kmph    0
pressure_hPa       0
visibility_km      0
dtype: int64


In [11]:
# Verify no missing values remain
print(df.isna().sum())

timestamp          0
temperature_C      0
humidity_%         0
wind_speed_kmph    0
pressure_hPa       0
visibility_km      0
dtype: int64


In [14]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour

# 1. Daily average temperature
daily_avg_temp = df.groupby('date')['temperature_C'].mean()
print("Daily Average Temperature:")
print(daily_avg_temp)


Daily Average Temperature:
date
2023-03-01    21.263158
2023-03-02    21.258991
2023-03-03    21.304825
2023-03-04    21.425658
2023-03-05    21.529825
2023-03-06    21.858333
2023-03-07    21.179825
2023-03-08    20.947807
2023-03-09    20.792325
2023-03-10    21.597149
Name: temperature_C, dtype: float64


In [17]:
# 2. Max, min, mean for each metric
metrics = ['temperature_C', 'humidity_%', 'wind_speed_kmph', 'pressure_hPa', 'visibility_km']
summary = df[metrics].agg(['max', 'min', 'mean'])
print("Summary Statistics:")
print(summary)

Summary Statistics:
      temperature_C  humidity_%  wind_speed_kmph  pressure_hPa  visibility_km
max       28.700000   88.100000         17.80000   1027.000000      12.600000
min       11.500000   47.800000          1.30000    998.100000       6.800000
mean      21.315789   66.795982         10.10531   1011.884753       9.989474


In [18]:
# 3. Most humid hour on average
avg_humidity_by_hour = df.groupby('hour')['humidity_%'].mean()
most_humid_hour = avg_humidity_by_hour.idxmax()
print(f"\nMost humid hour on average: {most_humid_hour}:00 with {avg_humidity_by_hour.max():.2f}% humidity")


Most humid hour on average: 1:00 with 78.42% humidity


In [19]:
# TODO: Extract temperature and wind_speed as NumPy arrays
import numpy as np

temp = df['temperature_C'].to_numpy()
wind = df['wind_speed_kmph'].to_numpy()

In [20]:
temp_matrix = temp.reshape((10, 24))

In [23]:
daily_min = temp_matrix.min(axis=1)
daily_max = temp_matrix.max(axis=1)
daily_mean = temp_matrix.mean(axis=1)

print("Daily Min Temperatures:", daily_min)
print("Daily Max Temperatures:", daily_max)
print("Daily Mean Temperatures:", daily_mean)

Daily Min Temperatures: [14.7 15.7 13.6 15.9 12.4 15.5 15.3 13.5 14.3 11.5]
Daily Max Temperatures: [28.2 28.7 25.7 27.1 24.9 26.2 25.9 26.  27.1 28.5]
Daily Mean Temperatures: [21.26315789 21.25899123 21.30482456 21.42565789 21.52982456 21.85833333
 21.17982456 20.94780702 20.79232456 21.59714912]


In [24]:
# TODO: Normalize temp_matrix
def normalize(matrix):
    return (matrix - matrix.mean()) / matrix.std()

# Apply it to temp_matrix
norm_matrix = normalize(temp_matrix)

print("Normalized matrix:")
print(norm_matrix)
print("\nCheck - Mean should be ~0:", norm_matrix.mean())
print("Check - Std should be ~1:", norm_matrix.std())

Normalized matrix:
[[-1.4173072  -1.53752522 -1.80801577 -1.65774325 -0.12496347 -0.15501798
   0.44607213  0.35590862 -0.03479995  2.06901543  0.          1.28759828
   1.22748927  0.62639917  1.88868839  0.74661719  0.20563609  0.71656268
   0.53623565  0.2356906  -0.93643512 -0.57578105 -0.48561753 -1.98834281]
 [-1.62768874 -1.68779775 -1.11676215 -0.39545402  0.20563609 -0.03479995
   0.41601763  0.05536356  0.68650818  2.21928795 -0.24518149  1.13732576
   0.2356906   0.95699873  0.38596312  0.17558158  0.77667169  0.
   0.86683521  0.20563609 -0.42550852 -0.75610808 -1.44736171 -0.99654413]
 [-2.31894237 -1.68779775 -0.8462716  -1.62768874  1.31765279 -0.12496347
  -0.15501798 -0.18507248  0.29579961  1.31765279  1.04716224 -0.03479995
   1.25754378  0.17558158  0.32585411  1.01710774  0.92694422  0.
   0.59634466  0.92694422  0.41601763 -0.12496347 -0.81621709 -1.77796127]
 [-1.3872527  -0.78616259 -1.02659863 -1.59763424  0.05536356  0.2356906
   1.73841587  0.71656268  0.1755

In [25]:
# TODO: Create boolean mask and filter wind speeds
mask = wind > 15
high_wind = wind[mask]

print("Total readings:", len(wind))
print("Number of high-wind readings (>15 kmph):", len(high_wind))
print("\nHigh wind speed values:")
print(high_wind)

Total readings: 240
Number of high-wind readings (>15 kmph): 30

High wind speed values:
[17.6 16.  16.5 16.3 16.7 15.8 17.8 15.1 16.3 15.2 17.  15.9 15.6 15.8
 15.4 15.6 16.3 15.3 16.2 16.9 15.3 15.2 15.5 17.4 17.4 15.4 15.4 16.5
 17.  15.7]


In [26]:
# TODO: Write and test your function
def daily_summary(matrix):
    summaries = []
    for i in range(matrix.shape[0]):
        day_data = matrix[i]
        summary = {
            'day': i + 1,
            'min': day_data.min(),
            'max': day_data.max(),
            'mean': day_data.mean()
        }
        summaries.append(summary)
    return summaries

# Example usage:
summaries = daily_summary(temp_matrix)
for s in summaries:
    print(s)

{'day': 1, 'min': np.float64(14.7), 'max': np.float64(28.2), 'mean': np.float64(21.263157894736846)}
{'day': 2, 'min': np.float64(15.7), 'max': np.float64(28.7), 'mean': np.float64(21.258991228070176)}
{'day': 3, 'min': np.float64(13.6), 'max': np.float64(25.7), 'mean': np.float64(21.304824561403507)}
{'day': 4, 'min': np.float64(15.9), 'max': np.float64(27.1), 'mean': np.float64(21.425657894736844)}
{'day': 5, 'min': np.float64(12.4), 'max': np.float64(24.9), 'mean': np.float64(21.52982456140351)}
{'day': 6, 'min': np.float64(15.5), 'max': np.float64(26.2), 'mean': np.float64(21.858333333333334)}
{'day': 7, 'min': np.float64(15.3), 'max': np.float64(25.9), 'mean': np.float64(21.17982456140351)}
{'day': 8, 'min': np.float64(13.5), 'max': np.float64(26.0), 'mean': np.float64(20.947807017543862)}
{'day': 9, 'min': np.float64(14.3), 'max': np.float64(27.1), 'mean': np.float64(20.792324561403507)}
{'day': 10, 'min': np.float64(11.5), 'max': np.float64(28.5), 'mean': np.float64(21.597149122